# Redis Storage Bridge — Validation

Tests the `hllset-storage-redis` crate: connects to the Redis container,
runs all `Storage` trait methods, and verifies HLLSet round-trip through
Redis. Requires Redis container running on port 6379.

In [2]:
:dep hllset-storage-redis = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-storage-redis" }
:dep hllset-storage = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-storage" }
:dep hllset-dsl = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-dsl" }
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/hllset-next/crates/hllset-core" }

In [6]:
use hllset_storage_redis::RedisStorage;
use hllset_storage::Storage;
use hllset_dsl::LatticeElement;

// Connect to Redis
let store = match RedisStorage::connect("redis://127.0.0.1:6379") {
    Ok(s) => s,
    Err(e) => {
        println!("ERROR: Cannot connect to Redis: {}", e);
        println!("Start the container: podman run -d --name hllset-redis -p 6379:6379 hllset-redis");
        return;
    }
};

println!("Connected to Redis at {}", store.url());
println!("PING: {:?}", store.ping());

Connected to Redis at redis://127.0.0.1:6379
PING: Ok("PONG")


## Basic CRUD — Store, Load, Exists, Delete

In [7]:
let key = "h:nb_test_crud";
let data = b"hello from redis notebook";

// Store
store.store(key, data).unwrap();
println!("Stored '{}' -> {}", key, String::from_utf8_lossy(data));

// Exists
println!("exists: {}", store.exists(key).unwrap());

// Load
let loaded = store.load(key).unwrap();
println!("loaded: {:?}", loaded.as_ref().map(|b| String::from_utf8_lossy(b)));
assert_eq!(loaded, Some(data.to_vec()));

// Delete
let deleted = store.delete(key).unwrap();
println!("deleted: {}", deleted);
println!("exists after delete: {}", store.exists(key).unwrap());

println!("\nCRUD test passed");

Stored 'h:nb_test_crud' -> hello from redis notebook
exists: true
loaded: Some("hello from redis notebook")
deleted: true
exists after delete: false

CRUD test passed


## List by Prefix

In [8]:
// Store test keys
for name in &["alpha", "beta", "gamma"] {
    store.store(&format!("h:nb_{}", name), name.as_bytes()).unwrap();
}
for name in &["delta", "epsilon"] {
    store.store(&format!("c:nb_{}", name), name.as_bytes()).unwrap();
}

let h_keys = store.list("h:nb_").unwrap();
let c_keys = store.list("c:nb_").unwrap();

println!("h: keys (expect 4 — crud + alpha/beta/gamma):");
for k in &h_keys { println!("  {}", k); }
println!("\nc: keys (expect 2 — delta/epsilon):");
for k in &c_keys { println!("  {}", k); }

assert!(h_keys.contains(&"h:nb_alpha".to_string()));
assert!(c_keys.contains(&"c:nb_delta".to_string()));

// Cleanup
for name in &["alpha", "beta", "gamma"] {
    store.delete(&format!("h:nb_{}", name)).unwrap();
}
for name in &["delta", "epsilon"] {
    store.delete(&format!("c:nb_{}", name)).unwrap();
}

println!("\nList test passed");

h: keys (expect 4 — crud + alpha/beta/gamma):
  h:nb_alpha
  h:nb_gamma
  h:nb_beta

c: keys (expect 2 — delta/epsilon):
  c:nb_epsilon
  c:nb_delta

List test passed


## Pin and Garbage Collection

In [9]:
store.store("h:nb_pin_keep", b"keep me").unwrap();
store.store("h:nb_pin_toss", b"toss me").unwrap();
store.pin("h:nb_pin_keep").unwrap();

let removed = store.gc().unwrap();
println!("GC removed: {:?}", removed);

assert!(removed.contains(&"h:nb_pin_toss".to_string()));
assert!(!removed.contains(&"h:nb_pin_keep".to_string()));
assert!(store.exists("h:nb_pin_keep").unwrap());
assert!(!store.exists("h:nb_pin_toss").unwrap());

// Cleanup
store.unpin("h:nb_pin_keep").unwrap();
store.delete("h:nb_pin_keep").unwrap();

println!("Pin/GC test passed");

GC removed: ["h:nb_pin_toss"]
Pin/GC test passed


## HLLSet Round-Trip — Tokenize → Store → Load → Verify

In [10]:
// Create an HLLSet from tokens
let tokens = &["machine", "learning", "neural", "network", "deep"];
let elem = LatticeElement::from_tokens(tokens);
let key = elem.key().to_string();
let bytes = elem.to_bytes();

println!("Created HLLSet:");
println!("  key:   {}", key);
println!("  size:  {} bytes", bytes.len());
println!("  popcount: {}", elem.popcount());
println!("  cardinality: {:.0}", elem.cardinality());

// Store in Redis
store.store(&key, &bytes).unwrap();
println!("\nStored in Redis: {}", store.exists(&key).unwrap());

// Load from Redis
let loaded_bytes = store.load(&key).unwrap().unwrap();
let restored = LatticeElement::from_bytes(&loaded_bytes).unwrap();

println!("\nRestored from Redis:");
println!("  key:   {}", restored.key());
println!("  popcount: {}", restored.popcount());

// Verify identity
assert_eq!(elem.key(), restored.key());
assert_eq!(elem.popcount(), restored.popcount());

// Verify lattice operations still work on restored element
let elem2 = LatticeElement::from_tokens(&["deep", "learning", "gradient"]);
let intersection = restored.intersection(&elem2);
let bss = restored.bss_inclusion(&elem2);
println!("\nLattice operations on restored HLLSet:");
println!("  intersect with 'deep learning gradient': popcount={}", intersection.popcount());
println!("  BSS inclusion: {:.3}", bss);

// Cleanup
store.delete(&key).unwrap();

println!("\nHLLSet round-trip test passed");

Created HLLSet:
  key:   h:4e1901f21b6209ed4325fb69b73707be945f55a4
  size:  26 bytes
  popcount: 5
  cardinality: 5

Stored in Redis: true

Restored from Redis:
  key:   h:4e1901f21b6209ed4325fb69b73707be945f55a4
  popcount: 5

Lattice operations on restored HLLSet:
  intersect with 'deep learning gradient': popcount=2
  BSS inclusion: 0.667

HLLSet round-trip test passed


use std::time::Instant;

let texts = vec![
    "Hello world from Redis backend",
    "HLLSet algebra content addressed storage",
    "machine learning neural network deep gradient",
    "Forth DSL FPGA native lattice operations",
    "Noether steering Fisher matrix rank derivatives",
];

let start = Instant::now();
let mut keys: Vec<String> = Vec::new();

for text in &texts {
    let tokens: Vec<&str> = text.split_whitespace().collect();
    let elem = LatticeElement::from_tokens(&tokens);
    let key = elem.key().to_string();
    let bytes = elem.to_bytes();
    store.store(&key, &bytes).unwrap();
    keys.push(key);
}

let elapsed = start.elapsed();
println!("Ingested {} texts in {:.1}ms", texts.len(), elapsed.as_secs_f64() * 1000.0);

let all_h = store.list("h:").unwrap();
println!("Total h: keys in Redis: {}", all_h.len());
println!("Our keys stored: {}", keys.len());

// Cleanup
for key in &keys {
    let _ = store.delete(key.as_str());
}

println!("\nBatch ingest test passed");

In [11]:
use std::time::Instant;

let texts = vec![
    "Hello world from Redis backend",
    "HLLSet algebra content addressed storage",
    "machine learning neural network deep gradient",
    "Forth DSL FPGA native lattice operations",
    "Noether steering Fisher matrix rank derivatives",
];

let start = Instant::now();
let mut keys: Vec<String> = Vec::new();

for text in &texts {
    let tokens: Vec<&str> = text.split_whitespace().collect();
    let elem = LatticeElement::from_tokens(&tokens);
    let key = elem.key().to_string();
    let bytes = elem.to_bytes();
    store.store(&key, &bytes).unwrap();
    keys.push(key);
}

let elapsed = start.elapsed();
println!("Ingested {} texts in {:.1}ms", texts.len(), elapsed.as_secs_f64() * 1000.0);

let all_h = store.list("h:").unwrap();
println!("Total h: keys in Redis: {}", all_h.len());
println!("Our keys stored: {}", keys.len());

// Cleanup
for key in &keys {
    let _ = store.delete(key.as_str());
}

println!("\nBatch ingest test passed");

Ingested 5 texts in 0.8ms
Total h: keys in Redis: 5
Our keys stored: 5

Batch ingest test passed


## Summary

All `Storage` trait methods verified against live Redis:

| Method | Redis command | Result |
|--------|-------------|--------|
| `store` | SET | Binary data stored and retrievable |
| `load` | GET | Full fidelity round-trip |
| `exists` | EXISTS | Returns true/false correctly |
| `delete` | DEL | Removes key, returns correct count |
| `list` | SCAN + MATCH | Prefix filtering works |
| `pin` | SADD hllset:pins | Pins a key |
| `unpin` | SREM hllset:pins | Unpins a key |
| `gc` | SCAN + DEL | Removes unpinned, keeps pinned |

HLLSet round-trip through Redis preserves identity, popcount, and lattice
operation semantics. The Redis backend is a drop-in replacement for
`MemoryStorage` and `IpfrsNativeStorage`.